# Evaluation

In [5]:
from lxml import etree
from rapidfuzz import fuzz
from pathlib import Path
import json

# =====================================================
# UTILS
# =====================================================

def normalize(text):
    return text.lower().strip() if text else ""


# =====================================================
# 1. PARSING TEI (ENTRIES)
# =====================================================

def parse_tei(xml_string: str):
    try:
        root = etree.fromstring(xml_string.encode())
    except:
        return

    ns = {"tei": root.nsmap.get(None)} if None in root.nsmap else {}

    def xp(node, path):
        return node.xpath(path, namespaces=ns)

    entries = []
    entry_path = ".//tei:entry" if ns else ".//entry"

    for entry in root.xpath(entry_path, namespaces=ns):
        lemma = xp(entry, ".//tei:form[@type='lemma']/tei:orth/text()" if ns else ".//form[@type='lemma']/orth/text()")
        pos = xp(entry, ".//tei:gramGrp/tei:pos/text()" if ns else ".//gramGrp/pos/text()")
        definition = xp(entry, ".//tei:sense/tei:def/text()" if ns else ".//sense/def/text()")

        entry_type = entry.get("type", "unknown")

        entries.append({
            "lemma": lemma[0].strip() if lemma else "",
            "pos": pos[0].strip() if pos else "",
            "definition": " ".join(definition).strip(),
            "entry_type": entry_type
        })

    return entries


# =====================================================
# 2. EXTRACTION GLOBALE DES BALISES
# =====================================================

def extract_global_tags(xml_string: str):
    root = etree.fromstring(xml_string.encode())

    ns = {"tei": root.nsmap.get(None)} if None in root.nsmap else {}

    def xp(path):
        return root.xpath(path, namespaces=ns)

    usg_dom = xp(".//tei:usg[@type='dom']" if ns else ".//usg[@type='dom']")
    usg_other = xp(".//tei:usg[not(@type='dom')]" if ns else ".//usg[not(@type='dom')]")
    milestones = xp(".//tei:milestone" if ns else ".//milestone")

    return {
        "usg_dom": len(usg_dom),
        "usg_other": len(usg_other),
        "milestone": len(milestones)
    }


# =====================================================
# 3. SPLIT PAR TYPE
# =====================================================

def split_by_type(entries):
    main = [e for e in entries if e["entry_type"] == "mainEntry"]
    related = [e for e in entries if e["entry_type"] == "relatedEntry"]
    return main, related


# =====================================================
# 4. ALIGNEMENT
# =====================================================

def align_entries(pred, gold, threshold=85):
    matches = []
    replacements = []
    insertions = []
    deletions = []

    used_gold = set()
    unmatched_pred = []

    # PASS 1 — exact lemma
    for p in pred:
        found = False
        for i, g in enumerate(gold):
            if i in used_gold:
                continue

            if normalize(p["lemma"]) == normalize(g["lemma"]):
                matches.append((p, g))
                used_gold.add(i)
                found = True
                break

        if not found:
            unmatched_pred.append(p)

    unmatched_gold = [g for i, g in enumerate(gold) if i not in used_gold]

    # PASS 2 — fuzzy
    used_gold_fuzzy = set()

    for p in unmatched_pred:
        best_match = None
        best_score = 0
        best_idx = None

        for i, g in enumerate(unmatched_gold):
            if i in used_gold_fuzzy:
                continue

            lemma_score = fuzz.ratio(normalize(p["lemma"]), normalize(g["lemma"]))
            def_score = fuzz.ratio(normalize(p["definition"]), normalize(g["definition"]))

            score = 0.5 * lemma_score + 0.5 * def_score

            if score > best_score:
                best_score = score
                best_match = g
                best_idx = i

        if best_score >= threshold:
            replacements.append((p, best_match))
            used_gold_fuzzy.add(best_idx)
        else:
            insertions.append(p)

    deletions = [
        g for i, g in enumerate(unmatched_gold)
        if i not in used_gold_fuzzy
    ]

    return {
        "matches": matches,
        "replacements": replacements,
        "insertions": insertions,
        "deletions": deletions
    }


# =====================================================
# 5. SCORING ENTRIES
# =====================================================

def compute_scores(alignment):
    tp = len(alignment["matches"])
    fp = len(alignment["insertions"]) + len(alignment["replacements"])
    fn = len(alignment["deletions"]) + len(alignment["replacements"])

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    }


# =====================================================
# 6. SCORING BALISES GLOBALES
# =====================================================

def compute_tag_scores(pred_tags, gold_tags):
    scores = {}

    for tag in ["usg_dom", "usg_other", "milestone"]:
        pred = pred_tags[tag]
        gold = gold_tags[tag]

        tp = min(pred, gold)
        fp = max(0, pred - gold)
        fn = max(0, gold - pred)

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

        scores[tag] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "pred_count": pred,
            "gold_count": gold
        }

    return scores


# =====================================================
# 7. PIPELINE GLOBAL
# =====================================================

def evaluation(pred_path, gold_path, output_path):
    pred_xml = Path(pred_path).read_text(encoding="utf-8")
    gold_xml = Path(gold_path).read_text(encoding="utf-8")

    # --- ENTRIES ---
    pred_entries = parse_tei(pred_xml)
    gold_entries = parse_tei(gold_xml)

    if pred_entries is None or gold_entries is None:
        print(f"    ⚠️  Parsing échoué pour {pred_path} ou {gold_path}, skip.")
        return None

    pred_main, pred_related = split_by_type(pred_entries)
    gold_main, gold_related = split_by_type(gold_entries)

    alignment_main = align_entries(pred_main, gold_main)
    alignment_related = align_entries(pred_related, gold_related)

    scores_main = compute_scores(alignment_main)
    scores_related = compute_scores(alignment_related)

    # --- TAGS ---
    pred_tags = extract_global_tags(pred_xml)
    gold_tags = extract_global_tags(gold_xml)

    tag_scores = compute_tag_scores(pred_tags, gold_tags)

    # --- FINAL REPORT ---
    final_report = {
        "entries": {
            "mainEntry": scores_main,
            "relatedEntry": scores_related
        },
        "tags": tag_scores
    }

    # --- SAVE ---
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    return final_report

In [6]:



"""
Project structure:

├── input/            # fichiers d'entrée
│   └── TR5_p489-490.txt
├── prompts/          # prompts utilisés
│   ├── Prompt_1.txt
│   ├── Prompt_2.txt
│   └── Prompt_3.txt
├── gt/               # ground truth
│   └── TR5_p489-490.xml
├── output/           # sorties des modèles
│   ├── gpt-5-mini/
│   │   └── Prompt_1/
│   │   │   └── TR5_p489-490.xml
│   │   └── Prompt_2/
│   │       └── TR5_p489-490.xml
│   └── gemma-3-27b-it/
│       └── Prompt_1/
│           └── TR5_p489-490.xml
│       └── Prompt_2/
│           └── TR5_p489-490.xml
└── evaluation/       # métriques / résultats d'évaluation
"""

from pathlib import Path

prompts = sorted(Path("prompts").glob("*.txt"))
models = sorted([p for p in Path("output").iterdir() if p.is_dir()])
inputs = sorted(Path("input").glob("*.txt"))

for input_file in inputs:
    doc_name = input_file.stem

    gt_file = Path("gt") / f"{doc_name}.xml"
    print(f"\n=== DOCUMENT: {doc_name} ===")

    for prompt in prompts:
        print(f"  → PROMPT: {prompt.stem}")

        for model_dir in models:
            output_file = model_dir / prompt.stem / f"{doc_name}.xml"
            
            status = "OK" if output_file.exists() else "MISSING"

            print(f"     → {model_dir.name}")

            evaluation(output_file, gt_file, output_path=f"evaluation/{model_dir.name}_{prompt.stem}_{doc_name}.json")


=== DOCUMENT: TR1_p2001-2002 ===
  → PROMPT: FS-R
     → gemini-3.1-flash-lite-preview
     → gemma-3-27b-it
     → gpt-5-mini
     → gpt-5.4-mini
  → PROMPT: FS
     → gemini-3.1-flash-lite-preview
     → gemma-3-27b-it
     → gpt-5-mini
     → gpt-5.4-mini
  → PROMPT: ZS
     → gemini-3.1-flash-lite-preview
     → gemma-3-27b-it
     → gpt-5-mini
     → gpt-5.4-mini
    ⚠️  Parsing échoué pour output/gpt-5.4-mini/ZS/TR1_p2001-2002.xml ou gt/TR1_p2001-2002.xml, skip.

=== DOCUMENT: TR1_p453-454 ===
  → PROMPT: FS-R
     → gemini-3.1-flash-lite-preview
     → gemma-3-27b-it
     → gpt-5-mini
     → gpt-5.4-mini
  → PROMPT: FS
     → gemini-3.1-flash-lite-preview
     → gemma-3-27b-it
     → gpt-5-mini
     → gpt-5.4-mini
  → PROMPT: ZS
     → gemini-3.1-flash-lite-preview
     → gemma-3-27b-it
    ⚠️  Parsing échoué pour output/gemma-3-27b-it/ZS/TR1_p453-454.xml ou gt/TR1_p453-454.xml, skip.
     → gpt-5-mini
     → gpt-5.4-mini

=== DOCUMENT: TR2_p1785-1786 ===
  → PROMPT: FS-R
     